# Lung Cancer Survey Classification
The project predicts lung cancer status from survey features and compares multiple classification models using stratified cross-validation.


## 1. Imports

The project uses pandas and NumPy for data handling, seaborn and matplotlib for visualization, scikit-learn for preprocessing, feature selection, models, and metrics, and XGBoost as an additional ensemble classifier.


In [ ]:
from __future__ import annotations
import argparse
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COLUMN = "LUNG_CANCER"


## 2. Load Dataset

The dataset is included as `data/survey_lung_cancer.csv`. The target column is `LUNG_CANCER`, with values `YES` and `NO`.


In [ ]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the lung cancer survey dataset."""

    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")
    return pd.read_csv(data_path)


In [ ]:
data_path = Path("data") / "survey_lung_cancer.csv"
raw_df = load_dataset(data_path)
print("Dataset shape:", raw_df.shape)
raw_df.head()


## 3. Initial Data Inspection

Inspect shape, data types, missing values, class balance, and basic descriptive statistics.


In [ ]:
raw_df.info()


In [ ]:
raw_df.isna().sum()


In [ ]:
raw_df.describe(include="all")


In [ ]:
raw_df["LUNG_CANCER"].value_counts()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=raw_df, x="LUNG_CANCER", color="#4f8fd6")
plt.title("Target Class Distribution")
plt.tight_layout()
plt.show()


## 4. Cleaning, Encoding, and Outlier Capping

Column names are stripped of accidental whitespace. Numerical outliers are capped with the IQR rule instead of dropping rows. Categorical variables are label encoded.


In [ ]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Strip accidental whitespace from column names."""

    clean_df = df.copy()
    clean_df.columns = clean_df.columns.str.strip()
    return clean_df

def cap_iqr_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Cap numerical outliers to IQR bounds without dropping rows."""

    capped_df = df.copy()
    numeric_cols = capped_df.select_dtypes(include=[np.number]).columns
    for column in numeric_cols:
        q1 = capped_df[column].quantile(0.25)
        q3 = capped_df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        capped_df[column] = capped_df[column].clip(lower_bound, upper_bound)
    return capped_df

def encode_categorical(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, LabelEncoder]]:
    """Label encode categorical columns."""

    encoded_df = df.copy()
    encoders = {}
    for column in encoded_df.select_dtypes(include=["object"]).columns:
        encoder = LabelEncoder()
        encoded_df[column] = encoder.fit_transform(encoded_df[column])
        encoders[column] = encoder
    return encoded_df, encoders


In [ ]:
clean_df = clean_column_names(raw_df)
capped_df = cap_iqr_outliers(clean_df)
encoded_df, encoders = encode_categorical(capped_df)
encoded_df.head()


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(encoded_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Encoded Feature Correlation Matrix")
plt.tight_layout()
plt.show()


## 5. Train/Test Split, Scaling, and Feature Selection

The data is split with stratification to preserve class balance. Features are scaled, and the top 8 features are selected with ANOVA F-test.


In [ ]:
def prepare_features(
    df: pd.DataFrame,
    num_features: int = 8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str], pd.DataFrame]:
    """Encode, split, scale, and select the strongest features."""

    clean_df = clean_column_names(df)
    capped_df = cap_iqr_outliers(clean_df)
    encoded_df, _ = encode_categorical(capped_df)

    x = encoded_df.drop(columns=[TARGET_COLUMN])
    y = encoded_df[TARGET_COLUMN]

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    selector = SelectKBest(score_func=f_classif, k=num_features)
    x_train_selected = selector.fit_transform(x_train_scaled, y_train)
    x_test_selected = selector.transform(x_test_scaled)
    selected_features = list(x.columns[selector.get_support(indices=True)])

    feature_scores = (
        pd.DataFrame({"feature": x.columns, "score": selector.scores_})
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

    return (
        x_train_selected,
        x_test_selected,
        y_train.to_numpy(),
        y_test.to_numpy(),
        selected_features,
        feature_scores,
    )


In [ ]:
(
    X_train_selected,
    X_test_selected,
    y_train,
    y_test,
    selected_features,
    feature_scores,
) = prepare_features(raw_df, num_features=8)

print("Selected features:", selected_features)
print("Train shape:", X_train_selected.shape)
print("Test shape:", X_test_selected.shape)
feature_scores


## 6. Model Definitions and Cross-Validation

The project compares tree-based, linear, distance-based, probabilistic, boosting, SVM, neural-network, and XGBoost classifiers.


In [ ]:
def build_models() -> dict[str, object]:
    """Create the classifiers used for comparison."""

    return {
        "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=RANDOM_STATE,
        ),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "GaussianNB": GaussianNB(),
        "SVC_linear": SVC(kernel="linear", random_state=RANDOM_STATE),
        "SVC_rbf": SVC(kernel="rbf", random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "LDA": LinearDiscriminantAnalysis(),
        "XGBoost": XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE,
        ),
        "MLP": MLPClassifier(max_iter=300, random_state=RANDOM_STATE),
    }

def cross_validate_models(
    models: dict[str, object],
    x: np.ndarray,
    y: np.ndarray,
    folds: int = 5,
) -> dict[str, dict[str, float]]:
    """Evaluate models with stratified cross-validation."""

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    for model_name, model in models.items():
        metrics = {"accuracy": [], "precision": [], "recall": [], "f1": []}
        for train_index, test_index in skf.split(x, y):
            x_train, x_test = x[train_index], x[test_index]
            y_train, y_test = y[train_index], y[test_index]

            model.fit(x_train, y_train)
            y_pred = model.predict(x_test)

            metrics["accuracy"].append(accuracy_score(y_test, y_pred))
            metrics["precision"].append(
                precision_score(y_test, y_pred, zero_division=0)
            )
            metrics["recall"].append(recall_score(y_test, y_pred, zero_division=0))
            metrics["f1"].append(f1_score(y_test, y_pred, zero_division=0))

        results[model_name] = {
            f"{metric}_mean": float(np.mean(values))
            for metric, values in metrics.items()
        }
        results[model_name].update(
            {
                f"{metric}_std": float(np.std(values, ddof=1))
                for metric, values in metrics.items()
            }
        )
    return results

def save_cv_plot(cv_results: dict[str, dict[str, float]], results_dir: Path) -> None:
    """Save a model accuracy comparison plot."""

    summary = pd.DataFrame.from_dict(cv_results, orient="index").reset_index()
    summary = summary.rename(columns={"index": "model"})

    plt.figure(figsize=(12, 6))
    sns.barplot(data=summary, x="model", y="accuracy_mean", color="#4f8fd6")
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("Mean CV Accuracy")
    plt.title("Model Cross-Validation Accuracy Comparison")
    plt.tight_layout()
    plt.savefig(results_dir / "cross_validation_accuracy.png", dpi=160)
    plt.close()

def print_summary(cv_results: dict[str, dict[str, float]]) -> None:
    """Print a compact sorted summary."""

    summary = pd.DataFrame.from_dict(cv_results, orient="index")
    summary = summary.sort_values(["accuracy_mean", "f1_mean"], ascending=False)
    print(summary[["accuracy_mean", "accuracy_std", "f1_mean", "f1_std"]].to_string())


## 7. Train and Compare Models

A 5-fold stratified cross-validation is run on the selected feature matrix. Metrics include accuracy, precision, recall, and F1-score.


In [ ]:
X_full = np.vstack([X_train_selected, X_test_selected])
y_full = np.hstack([y_train, y_test])

models = build_models()
cv_results = cross_validate_models(models, X_full, y_full, folds=5)
print_summary(cv_results)


## 8. Save Generated Results

When this notebook is run, generated metrics and feature scores are saved in the `results/` folder.


In [ ]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

feature_scores.to_csv(results_dir / "feature_scores.csv", index=False)
save_cv_plot(cv_results, results_dir)

output = {
    "dataset_shape": list(raw_df.shape),
    "selected_features": selected_features,
    "cross_validation": cv_results,
}
with (results_dir / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(output, file, indent=2)

print(f"Saved generated results to: {results_dir}")


## 9. Final Recorded Results

The table below is copied from the original submitted notebook outputs.

| Model | Accuracy Mean | Accuracy Std | F1 Mean | F1 Std |
|---|---:|---:|---:|---:|
| DecisionTree | 0.8925 | 0.0091 | 0.9382 | 0.0059 |
| RandomForest | **0.9056** | 0.0177 | 0.9466 | 0.0100 |
| ExtraTrees | 0.8958 | 0.0088 | 0.9407 | 0.0052 |
| AdaBoost | 0.8992 | 0.0204 | 0.9443 | 0.0105 |
| GradientBoosting | 0.8990 | 0.0134 | 0.9425 | 0.0080 |
| GaussianNB | 0.8892 | 0.0079 | 0.9372 | 0.0049 |
| SVC linear | 0.8926 | 0.0238 | 0.9408 | 0.0121 |
| SVC rbf | 0.9024 | 0.0192 | 0.9455 | 0.0105 |
| KNN | 0.8959 | 0.0266 | 0.9419 | 0.0136 |
| XGBoost | **0.9056** | 0.0209 | **0.9468** | 0.0117 |
| MLP | **0.9056** | 0.0134 | **0.9468** | 0.0074 |

The highest recorded mean accuracy was shared by RandomForest, XGBoost, and MLP. XGBoost and MLP had the highest recorded mean F1-score.
